# Rodada B no Google Colab — BLIP 576 e BLIP sem fine-tuning

Mesmo protocolo da rodada A no Kaggle (ver `model/README.md`): mesma base limpa, divisão
80/10/10 por livro, semente 42, até 20 epochs com parada antecipada (paciência 3),
melhor checkpoint pela validação, mesmas métricas (`src/metrics.py`).

**Um modelo por sessão**, porque o Colab grátis derruba sessões longas. Cada sessão salva
`train.log`, `history.json`, `eval_val.log`, `eval_test.log` e `gpu.txt` em
`Meu Drive/tcc-resultados/<modelo>/` assim que termina. Se a sessão cair no meio, é só
rodar de novo o mesmo modelo.

Antes de rodar:
1. **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4.** (A rodada A
   rodou numa T4; outra GPU muda um pouco os números — a célula 1 avisa.)
2. **Chave do Kaggle** (as fotos vêm do dataset privado `pedrocanoas/fotos-livros`): no menu
   da esquerda, ícone de chave (**Secrets**) → **Add new secret** → nome `KAGGLE_API_TOKEN`,
   valor = o conteúdo de `C:\Users\Pedro\.kaggle\access_token` → ligar
   "Notebook access".
3. Escolher o modelo na célula 2 e **Ambiente de execução → Executar tudo**. Deixe a aba
   aberta até terminar.

In [ ]:
# Célula 1 — confere a GPU
import subprocess
import torch

assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução → Alterar o tipo → GPU T4"
GPU = (
    f"{torch.cuda.get_device_name(0)} (compute capability "
    f"{'.'.join(map(str, torch.cuda.get_device_capability(0)))}), torch {torch.__version__}"
)
print("GPU:", GPU)
if "T4" not in GPU:
    print("ATENÇÃO: a rodada A rodou numa T4. Com outra GPU os números mudam um pouco — registre no texto.")

In [ ]:
# Célula 2 — escolha o modelo desta sessão (um por vez)
MODELO = "blip_576_congelado"  # @param ["blip_576_congelado", "blip_576_2camadas", "blip_sem_finetuning"]

ARGS = {
    "blip_576_congelado": "--batch-size 8 --image-size 576",
    # batch 8 não coube na T4 (CUDA out of memory): batch 4 x 2 acumulações = mesmo batch efetivo
    "blip_576_2camadas": "--batch-size 4 --grad-accum 2 --image-size 576 --unfreeze-vision-layers 2 --vision-lr 1e-5",
    "blip_sem_finetuning": None,  # só avaliação do BLIP original
}
print("Modelo desta sessão:", MODELO)

In [ ]:
# Célula 3 — Google Drive (onde os resultados ficam guardados)
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/tcc-resultados") / MODELO
OUT.mkdir(parents=True, exist_ok=True)
if (OUT / "eval_test.log").exists():
    print(f"AVISO: {OUT} já tem resultados — rodar de novo vai sobrescrever.")
print("Resultados vão para:", OUT)

In [ ]:
# Célula 4 — código do GitHub + dependências
!rm -rf /content/repo && git clone --depth 1 https://github.com/Pedrocanoas/tcc.git /content/repo
%cd /content/repo/model
!git log --oneline -1
# kaggle >= 2 lê KAGGLE_API_TOKEN; six: o kaggle importa sem declarar como dependência
!pip install -q -r requirements-kaggle.txt "kaggle>=2.2" six

In [ ]:
# Célula 5 — fotos do dataset do Kaggle
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
RAW = Path("/content/fotos-livros")
if not (RAW / "livro001").exists():
    RAW.mkdir(exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "pedrocanoas/fotos-livros", "-p", str(RAW), "--unzip", "-q"], check=True)
n_books = len(list(RAW.glob("livro*")))
print(RAW, "-", n_books, "pastas de livro")
# 1929 pastas; 11 sem descrição válida são descartadas pelo prepare_dataset (fica 1918 livros)
assert n_books == 1929, "dataset diferente do usado nas rodadas do Kaggle"

os.environ["TCC_DATASET_RAW_DIR"] = str(RAW)
os.environ["TCC_DATASET_PROCESSED_DIR"] = "/content/processed"
os.environ["TCC_CHECKPOINTS_DIR"] = "/content/checkpoints"

In [ ]:
# Célula 6 — divisão 80/10/10 e conferência contra a rodada A
!python -m src.data.prepare_dataset --val-ratio 0.1 --test-ratio 0.1 --seed 42 --exclude-colecoes
counts = {s: sum(1 for _ in open(f"/content/processed/{s}.jsonl", encoding="utf-8")) for s in ("train", "val", "test")}
print(counts)
assert counts == {"train": 2215, "val": 309, "test": 300}, "divisão diferente da rodada A — não continue"


In [ ]:
# Célula 7 — treino (pula no BLIP sem fine-tuning) e avaliação na validação e no teste
import shutil

def sh(cmd):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")

(OUT / "gpu.txt").write_text(GPU + "\n")
COMMON = f"--raw-dir {RAW} --processed-dir /content/processed --epochs 20 --patience 3 --seed 42 --max-length 128 --exclude-colecoes"
CKPT = Path(f"/content/ckpt_{MODELO}")

if ARGS[MODELO] is None:
    checkpoint = "/content/nao_existe"  # src.evaluate cai no BLIP original
else:
    sh(
        f"set -o pipefail; python -u kaggle_train.py {COMMON} {ARGS[MODELO]} --checkpoints-dir {CKPT} 2>&1"
        f" | tee /content/train.log | grep --line-buffered -E 'resolu|trein|== epoch|parada|Error'"
    )
    shutil.copy("/content/train.log", OUT / "train.log")
    shutil.copy(CKPT / "history.json", OUT / "history.json")
    checkpoint = CKPT / "best"

for split in ("val", "test"):
    sh(f"python -u -m src.evaluate --checkpoint {checkpoint} --split {split} > /content/eval_{split}.log 2>&1")
    shutil.copy(f"/content/eval_{split}.log", OUT / f"eval_{split}.log")
    sh(f"tail -5 /content/eval_{split}.log")

print("\nPronto. Arquivos em", OUT)
!ls -la "{OUT}"